In [1]:
# Instalar Whisper, numpy 1.26.4 y FFmpeg (requiere git y ffmpeg ya instalados)
!pip install openai-whisper --quiet
!pip install numpy==1.26.4 --quiet
!pip install ffmpeg-python noisereduce transformers librosa jiwer --quiet

In [ ]:
import whisper

AUDIO_PATH = "audiotest.wav"

model = whisper.load_model("small")

result = model.transcribe(
    AUDIO_PATH,
    language="es",
    fp16=False,
    verbose=True,
    condition_on_previous_text=True,
    temperature=0.5,
    best_of=10
)

print("Transcripción completa:")
print(result['text'])


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: Spanish
[00:00.000 --> 00:04.360]  Dije, vaya vida, ha tenido ésta.
[00:04.360 --> 00:06.420]  ¿Qué cosas le han pasado?
[00:06.420 --> 00:08.800]  Dic que ella también ha tenido sus aventuras.
[00:08.800 --> 00:09.800]  ¿Yo?
[00:09.800 --> 00:14.760]  Un día se quedó dormida en el metro con su amiga Pili Morarte y les despertó un señor
[00:14.760 --> 00:18.020]  que resultó ser un jefazo pero de los gordos, de la UGT.
[00:20.020 --> 00:26.600]  Y cuando cortó con su novio será fin, dijo el otro, fin y dice ésta, fin, será fin.
[00:31.000 --> 00:33.000]  ¿Me dio un bajo?
[00:35.360 --> 00:38.480]  Una mala cara y sudando.
[00:38.480 --> 00:40.000]  No decía nada.
[00:40.000 --> 00:41.000]  No podía.
[00:42.000 --> 00:45.000]  Decíamos que querrá.
[00:45.000 --> 00:47.720]  Digo 20.000 euros y un vaso de agua.
[00:47.720 --> 00:50.120]  Dije, vago con el agua de momento.
[00:

In [5]:
from datetime import timedelta

def format_timestamp(seconds):
    return str(timedelta(seconds=int(seconds))) + "," + str(int((seconds % 1) * 1000)).zfill(3)

def export_srt(segments, filename="audiotest.srt"):
    with open(filename, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            start = format_timestamp(seg['start'])
            end = format_timestamp(seg['end'])
            text = seg['text'].strip()
            f.write(f"{i}\n{start} --> {end}\n{text}\n\n")

export_srt(result["segments"])
print("✅ Subtítulos exportados")


✅ Subtítulos exportados


In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_id = "dreuxx26/Multilingual-grammar-Corrector-using-mT5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def corregir_texto(texto):
    input_text = texto
    #input_text = texto
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = lm_model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Abrimos archivo SRT en modo escritura
with open("audiotest_corregido.srt", "w", encoding="utf-8") as f:
    for i, segment in enumerate(result['segments'], start=1):
        start_time = format_timestamp(segment['start'])
        end_time = format_timestamp(segment['end'])

        print(f"\n⏱️ [{segment['start']:.2f}s - {segment['end']:.2f}s]")
        original = segment['text']
        corregido = corregir_texto(original)
        print(f"Original: {original}")
        print(f"Corregido: {corregido}")

        # Escribir en archivo SRT
        f.write(f"{i}\n{start_time} --> {end_time}\n{corregido.strip()}\n\n")

print("✅ Proceso completado. Subtítulos corregidos guardados como 'audiotest_corregido.srt'")
 

tokenizer_config.json:   0%|          | 0.00/893 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/416 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


ValueError: Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.